# Scrap Serenity (@aleabitoreddit)

X(Twitter) 트윗 수집 → `analysis_Serenity.db`

- **post**: 일반 게시물
- **reply**: 답글
- **subscriber**: 구독자 전용 (`exclusivityInfo` 기반 감지)

## 1. Configuration

쿠키는 `.env` 파일에서 로드됩니다. 만료 시 `.env`의 `X_AUTH_TOKEN`, `X_CT0`, `X_TWID` 값을 교체하세요.

In [1]:
# ── Cookie ────────────────────────────────────────────────────────────────────
# X(Twitter) 쿠키 — 만료 시 .env 파일에서 새 값으로 교체
import os
from dotenv import load_dotenv

_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
load_dotenv(os.path.join(_notebook_dir, '.env'))

COOKIES = {
    "auth_token": os.environ["X_AUTH_TOKEN"],
    "ct0": os.environ["X_CT0"],
    "twid": os.environ["X_TWID"],
}

# ── Target ────────────────────────────────────────────────────────────────────
TARGET_USER = "aleabitoreddit"

# ── Options ───────────────────────────────────────────────────────────────────
TWEET_COUNT = None   # None = 전체, 숫자 = 최대 N개
DEBUG = False        # True = raw tweet._data JSON 덤프

# ── Paths ─────────────────────────────────────────────────────────────────────
DB_DIR = os.path.join(_notebook_dir, 'References')
DB_PATH = os.path.join(DB_DIR, 'analysis_Serenity.db')

os.makedirs(DB_DIR, exist_ok=True)
print(f"DB path: {DB_PATH}")

DB path: /Users/seongjin/Documents/⭐성진이의 옵시디언/💶Invest/.agents/skills/Serenity/References/analysis_Serenity.db


## 2. Setup

In [2]:
import asyncio
import sqlite3
import json
import re
import tempfile
from datetime import datetime, timedelta, timezone
from twikit import Client

# twikit 2.3.3 버그 우회: User.__init__이 legacy의 여러 키를 .get() 없이 직접 접근해 KeyError.
# X API 응답에서 누락 가능한 키들을 미리 기본값으로 채워 넣는다.
import twikit.user as _twikit_user
_orig_user_init = _twikit_user.User.__init__
_LEGACY_DEFAULTS = {
    'created_at': '', 'name': '', 'screen_name': '', 'profile_image_url_https': '',
    'location': '', 'description': '', 'pinned_tweet_ids_str': [],
    'verified': False, 'possibly_sensitive': False, 'can_dm': False, 'can_media_tag': False,
    'want_retweets': False, 'default_profile': False, 'default_profile_image': False,
    'has_custom_timelines': False, 'is_translator': False, 'translator_type': 'none',
    'followers_count': 0, 'fast_followers_count': 0, 'normal_followers_count': 0,
    'friends_count': 0, 'favourites_count': 0, 'listed_count': 0,
    'media_count': 0, 'statuses_count': 0, 'withheld_in_countries': [],
}
def _patched_user_init(self, client, data):
    data.setdefault('is_blue_verified', False)
    legacy = data.setdefault('legacy', {})
    for _k, _v in _LEGACY_DEFAULTS.items():
        legacy.setdefault(_k, _v)
    legacy.setdefault('entities', {}).setdefault('description', {}).setdefault('urls', [])
    _orig_user_init(self, client, data)
_twikit_user.User.__init__ = _patched_user_init

KST = timezone(timedelta(hours=9))
RATE_LIMIT_DELAY = 2
DUPLICATE_THRESHOLD = 10
MAX_RETRIES = 3


def to_kst(twitter_time_str):
    dt = datetime.strptime(twitter_time_str, "%a %b %d %H:%M:%S %z %Y")
    return dt.astimezone(KST).strftime("%Y-%m-%dT%H:%M:%S+09:00")


def extract_tickers(text):
    tickers = re.findall(r'\$([A-Za-z]+)', text)
    return list(dict.fromkeys([t.upper() for t in tickers]))


def get_full_content(tweet):
    note = (tweet._data
            .get('note_tweet', {})
            .get('note_tweet_results', {})
            .get('result', {}))
    if note and 'text' in note:
        return note['text']
    return tweet.full_text


def get_media_urls(tweet):
    urls = []
    if tweet.media:
        for m in tweet.media:
            if hasattr(m, 'media_url') and m.media_url:
                urls.append(m.media_url)
    return urls


def classify_tweet_type(tweet):
    if tweet._data.get('exclusivityInfo'):
        return 'subscriber'
    elif tweet.in_reply_to:
        return 'reply'
    else:
        return 'post'


def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS tweets (
            id TEXT PRIMARY KEY,
            user TEXT NOT NULL,
            type TEXT NOT NULL CHECK(type IN ('post', 'reply', 'subscriber')),
            created_at TEXT NOT NULL,
            content TEXT,
            tickers TEXT DEFAULT '[]',
            media TEXT DEFAULT '[]'
        )
    ''')
    conn.commit()
    return conn


def get_existing_ids(conn):
    cur = conn.execute('SELECT id FROM tweets')
    return set(row[0] for row in cur.fetchall())


def save_tweets(conn, tweets):
    saved = 0
    for tweet in tweets:
        content = get_full_content(tweet)
        tickers = json.dumps(extract_tickers(content), ensure_ascii=False)
        media = json.dumps(get_media_urls(tweet), ensure_ascii=False)
        tweet_type = classify_tweet_type(tweet)
        conn.execute(
            'INSERT OR IGNORE INTO tweets (id, user, type, created_at, content, tickers, media) VALUES (?,?,?,?,?,?,?)',
            (tweet.id, tweet.user.screen_name if tweet.user else TARGET_USER,
             tweet_type, to_kst(tweet.created_at), content, tickers, media))
        if conn.total_changes:
            saved += 1
    conn.commit()
    return saved


async def fetch_with_retry(coro_fn):
    for attempt in range(MAX_RETRIES):
        try:
            return await coro_fn()
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                raise
            wait = 2 ** (attempt + 1)
            print(f"  ⚠ {e} — retry in {wait}s...")
            await asyncio.sleep(wait)


print("Setup complete.")

Setup complete.


## 3. Scrape

In [3]:
async def run_scrape():
    conn = init_db()
    existing_ids = get_existing_ids(conn)
    print(f"DB: {DB_PATH}")
    print(f"Existing: {len(existing_ids)}")

    # 쿠키를 임시 파일로 저장하여 twikit에 전달
    cookie_file = tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False)
    json.dump(COOKIES, cookie_file)
    cookie_file.close()

    client = Client('en-US')
    try:
        client.load_cookies(cookie_file.name)
    except Exception as e:
        print(f"✗ Cookie expired — update .env (X_AUTH_TOKEN, X_CT0, X_TWID).\n  {e}")
        conn.close()
        return
    finally:
        os.unlink(cookie_file.name)

    user = await client.get_user_by_screen_name(TARGET_USER)
    print(f"\n=== {user.name} (@{user.screen_name}) ===")
    print(f"Followers: {user.followers_count:,} | Tweets: {user.statuses_count:,}")

    total_saved = 0
    type_counts = {'post': 0, 'reply': 0, 'subscriber': 0}

    for tweet_type in ['Tweets', 'Replies']:
        print(f"\nFetching {tweet_type}...")
        consecutive_dups = 0
        total_fetched = 0

        try:
            tweets = await fetch_with_retry(
                lambda tt=tweet_type: user.get_tweets(tt, count=20))
        except Exception as e:
            print(f"  ⚠ Failed: {e}")
            continue

        while tweets:
            page_tweets = []
            for t in tweets:
                sn = t.user.screen_name if t.user else None
                if sn and sn.lower() != TARGET_USER.lower():
                    continue

                total_fetched += 1

                if DEBUG:
                    print(f"  [DEBUG] {t.id}: exclusivityInfo={t._data.get('exclusivityInfo')}, in_reply_to={t.in_reply_to}")

                if t.id in existing_ids:
                    consecutive_dups += 1
                    if consecutive_dups >= DUPLICATE_THRESHOLD:
                        print(f"  {DUPLICATE_THRESHOLD} consecutive dups — stopping")
                        break
                else:
                    consecutive_dups = 0
                    tt = classify_tweet_type(t)
                    type_counts[tt] += 1
                    page_tweets.append(t)
                    existing_ids.add(t.id)
                    if TWEET_COUNT and (total_saved + len(page_tweets)) >= TWEET_COUNT:
                        break

            if page_tweets:
                saved = save_tweets(conn, page_tweets)
                total_saved += saved
                print(f"  Fetched {total_fetched}, saved {total_saved} total")

            if consecutive_dups >= DUPLICATE_THRESHOLD:
                break
            if TWEET_COUNT and total_saved >= TWEET_COUNT:
                break

            await asyncio.sleep(RATE_LIMIT_DELAY)
            try:
                tweets = await fetch_with_retry(lambda: tweets.next())
                if not tweets:
                    break
            except Exception as e:
                print(f"  ⚠ Pagination stopped: {e}")
                break

        if TWEET_COUNT and total_saved >= TWEET_COUNT:
            break

    print(f"\n{'='*40}")
    print(f"Saved this run: {total_saved}")
    print(f"  post={type_counts['post']}, reply={type_counts['reply']}, subscriber={type_counts['subscriber']}")

    # DB 통계
    cur = conn.execute('SELECT type, COUNT(*) FROM tweets GROUP BY type')
    stats = cur.fetchall()
    total = sum(c for _, c in stats)
    print(f"\nTotal in DB: {total}")
    for t, c in stats:
        print(f"  {t}: {c}")
    conn.close()

await run_scrape()

DB: /Users/seongjin/Documents/⭐성진이의 옵시디언/💶Invest/.agents/skills/Serenity/References/analysis_Serenity.db
Existing: 1585

=== Serenity (@aleabitoreddit) ===
Followers: 873,488 | Tweets: 7,403

Fetching Tweets...
  Fetched 19, saved 19 total
  10 consecutive dups — stopping
  Fetched 30, saved 20 total

Fetching Replies...
  Fetched 8, saved 21 total
  10 consecutive dups — stopping

Saved this run: 21
  post=19, reply=1, subscriber=1

Total in DB: 1606
  post: 1435
  reply: 2
  subscriber: 169


## 4. Explore DB

In [4]:
# ── 최신 트윗 확인 ─────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    'SELECT id, type, created_at, substr(content,1,80), tickers FROM tweets ORDER BY created_at DESC LIMIT 10')
for row in cur:
    print(f"[{row[1]:10s}] {row[2]}  {row[3]}")
    if row[4] != '[]':
        print(f"             tickers: {row[4]}")
conn.close()

[post      ] 2026-06-22T11:14:41+09:00  I expect Japan to mog Sweden in the World Cup tomorrow. 

Probably another 4-0 a
             tickers: ["SIVE"]
[reply     ] 2026-06-22T10:25:59+09:00  @troyofsparta Ajinomoto price hiked by 30% last month so Palliser actually succe
[post      ] 2026-06-22T10:20:35+09:00  Wow, 3 limit ups in a row with WUS TW. Pretty sad I didn't take larger positions
[post      ] 2026-06-22T09:55:20+09:00  Jeez, Japanese markets seem happy.

Everything from Furukawa, Towa, Harmonic Dri
[post      ] 2026-06-21T20:04:51+09:00  That’s a misconception:

$SIVE is the laser supplier for next gen architectures,
             tickers: ["SIVE", "JBL", "POET", "MRVL", "GFS"]
[post      ] 2026-06-21T16:53:18+09:00  I've been getting a lot of questions about OE Solutions (138080) recently. 

Her
             tickers: ["AAOI", "COHR", "LITE", "SIVE", "JBL", "GFS"]
[post      ] 2026-06-21T15:18:53+09:00  Apparently gym bros are the new hyperscalers.

They’ve created a new bott

In [5]:
# ── 타입별 통계 ───────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute('SELECT type, COUNT(*) FROM tweets GROUP BY type')
for t, c in cur:
    print(f"  {t}: {c}")
total = conn.execute('SELECT COUNT(*) FROM tweets').fetchone()[0]
print(f"  total: {total}")
conn.close()

  post: 1435
  reply: 2
  subscriber: 169
  total: 1606


In [6]:
# ── 구독자 전용 트윗 확인 ─────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    "SELECT created_at, substr(content,1,100), tickers FROM tweets WHERE type='subscriber' ORDER BY created_at DESC LIMIT 10")
for row in cur:
    print(f"{row[0]}  {row[1]}")
    if row[2] != '[]':
        print(f"  tickers: {row[2]}")
    print()
conn.close()

2026-06-21T10:09:19+09:00  This is the first time I’ve heard the name organ-on-chip and it sounds kinda gross.

Apparently this

2026-06-17T13:21:12+09:00  So I’m reading about an activist report about WUS (2316) right now from Palliser.

I actually though

2026-06-12T17:14:56+09:00  So doing research into some random $TSM, SK Hynix, SMIC, type bottleneck. 

not familiar with WF₆ bu
  tickers: ["TSM", "AXTI"]

2026-06-11T20:51:48+09:00  I'm trying a new research style going through trade records for fun, in case I find anything random.
  tickers: ["LPK"]

2026-06-11T15:30:29+09:00  Lot of $TSM CoPoS posts on X... just some of their likely external suppliers for this:

Favite(3535)
  tickers: ["TSM"]

2026-06-07T10:34:54+09:00  Looks like HBF (memory) wars are kicking off:

- Hanmi Semiconductor (first mover)
- Hanwha Semicond

2026-06-04T18:53:09+09:00  $TSM chairman: CoPoS very large within 2-3 year volumes already pilot lines now. 

Thought markets w
  tickers: ["TSM", "XFAB"]

2026-

In [7]:
# ── 특정 티커 검색 ────────────────────────────────────────────────────────
SEARCH_TICKER = "AXTI"  # ← 변경하여 검색

conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    "SELECT type, created_at, substr(content,1,100) FROM tweets WHERE tickers LIKE ? ORDER BY created_at DESC",
    (f'%"{SEARCH_TICKER}"%',))
rows = cur.fetchall()
print(f"${SEARCH_TICKER} mentions: {len(rows)}")
for row in rows[:10]:
    print(f"  [{row[0]}] {row[1]}  {row[2]}")
conn.close()

$AXTI mentions: 207
  [post] 2026-06-20T20:52:49+09:00  I think u must be new here. 

Many of my ideas get intense backlash at the start, especially the mor
  [post] 2026-06-19T21:19:38+09:00  I think something to highlight also is not all my ideas are green, especially on short term timefram
  [post] 2026-06-16T19:00:19+09:00  Yeah... I'm not quite sure why everyone likes just throwing personal accusations.

Like "scam" or  "
  [post] 2026-06-16T16:28:51+09:00  Fun throwback to random ideas back in 2025.

Back then, $AAOI was $2B MC, $LITE was a $26B MC, $AXTI
  [post] 2026-06-15T11:02:59+09:00  Today, there's a new report that China eased InP substrate exports. 

Which is expected to relieve m
  [post] 2026-06-15T00:27:22+09:00  This is gonna upset a lot of people: 

But TA is astrology for traders.

It's confirmation bias + tr
  [post] 2026-06-13T21:53:13+09:00  It’s been officially 3 months since I posted my $SIVE long thesis back at 4 SEK.

This idea is now u
  [post] 2026-06-13T1